# Watch Ridge Regression Pricer

Train and evaluate watch Ridge regression baselines with and without auction estimate features.

## Configuration

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "watch_ridge_pricer").exists():
    PROJECT_DIR = NOTEBOOK_DIR
else:
    PROJECT_DIR = Path("Linear_Regression/watch/Ridge_Regression").resolve()

sys.path.insert(0, str(PROJECT_DIR))

OUTPUT_DIR = PROJECT_DIR / "outputs"
WORKSPACE_ROOT = PROJECT_DIR.parents[2]
WATCH_DATA_DIR = WORKSPACE_ROOT / "Data" / "watch"
TRAIN_FILE = WATCH_DATA_DIR / "Watch_2026_finetune_19345.json"
QUARTERS = ("2025Q1", "2025Q2", "2025Q3", "2025Q4")
TEST_FILES = {
    quarter: WATCH_DATA_DIR / f"watch_sales_{quarter}_dedup_lot_url.json"
    for quarter in QUARTERS
}
TARGET_COL = "ah_price_usd"
TRAIN_BEFORE = "2025-01-01"
TRAIN_ROWS = None
TEST_ROWS_PER_QUARTER = None
MIN_CATEGORY_COUNT = 20
VALIDATION_FRACTION = 0.10
RANDOM_STATE = 42
VERBOSE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Train file:  {TRAIN_FILE}")
for quarter, path in TEST_FILES.items():
    print(f"Test file {quarter}: {path}")

Project dir: /srv/notebooks/[6.0] ML Pricers/Linear_Regression/watch/Ridge_Regression
Output dir:  /srv/notebooks/[6.0] ML Pricers/Linear_Regression/watch/Ridge_Regression/outputs
Train file:  /srv/notebooks/[6.0] ML Pricers/Data/watch/Watch_2026_finetune_19345.json
Test file 2025Q1: /srv/notebooks/[6.0] ML Pricers/Data/watch/watch_sales_2025Q1_dedup_lot_url.json
Test file 2025Q2: /srv/notebooks/[6.0] ML Pricers/Data/watch/watch_sales_2025Q2_dedup_lot_url.json
Test file 2025Q3: /srv/notebooks/[6.0] ML Pricers/Data/watch/watch_sales_2025Q3_dedup_lot_url.json
Test file 2025Q4: /srv/notebooks/[6.0] ML Pricers/Data/watch/watch_sales_2025Q4_dedup_lot_url.json


## Imports

In [2]:
import pandas as pd

from watch_ridge_pricer import (
    apply_feature_schema,
    build_feature_report,
    evaluate_by_price_bucket,
    evaluate_by_quarter,
    evaluate_price_predictions,
    fit_feature_schema,
    fit_ridge_log_price_model,
    get_top_coefficients,
    load_watch_data,
    make_model_frame,
    predict_price_from_log_model,
    ridge_training_history,
    split_train_test,
)

## Load and Split Data

In [3]:
missing_inputs = [path for path in [TRAIN_FILE, *TEST_FILES.values()] if not path.exists()]
if missing_inputs:
    raise FileNotFoundError("Missing input files:\n" + "\n".join(str(path) for path in missing_inputs))

raw = load_watch_data(
    train_file=TRAIN_FILE,
    test_dir=WATCH_DATA_DIR,
    train_before=TRAIN_BEFORE,
    quarters=QUARTERS,
    verbose=VERBOSE,
)
train_raw, test_raw = split_train_test(
    raw,
    train_rows=TRAIN_ROWS,
    test_rows_per_quarter=TEST_ROWS_PER_QUARTER,
    random_state=RANDOM_STATE,
    verbose=VERBOSE,
)

train_keys = set(train_raw["lot_url"].dropna().astype(str))
test_keys = set(test_raw["lot_url"].dropna().astype(str))
print(f"train_rows={len(train_raw):,}")
print(f"test_rows={len(test_raw):,}")
print(f"lot_url_overlap={len(train_keys & test_keys):,}")
test_raw["source_quarter"].value_counts().sort_index()

[watch features] Filtered train sale_date < 2025-01-01: 19,345->16,581
[watch features] Loaded watch_sales_2025Q1_dedup_lot_url.json: 574 rows, 79 columns
[watch features] Loaded watch_sales_2025Q2_dedup_lot_url.json: 1,681 rows, 79 columns
[watch features] Loaded watch_sales_2025Q3_dedup_lot_url.json: 483 rows, 79 columns
[watch features] Loaded watch_sales_2025Q4_dedup_lot_url.json: 2,257 rows, 79 columns


/srv/notebooks/[6.0] ML Pricers/Linear_Regression/watch/Ridge_Regression/watch_ridge_pricer/features.py:148: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True, sort=False)


[watch features] Loaded watch frame: 21,576 rows, 79 columns
[watch features] Initial split: train=16,581, explicit test=4,995
[watch features] De-duplicated train lot_url: 16,581->16,581
[watch features] De-duplicated test lot_url: 4,995->4,995
[watch features] Removed test-overlapping lot_url values from train: 16,581->16,581
[watch features] Final split: train=16,581, test=4,995
train_rows=16,581
test_rows=4,995
lot_url_overlap=0


source_quarter
2025Q1     574
2025Q2    1681
2025Q3     483
2025Q4    2257
Name: count, dtype: int64

## Train and Evaluate Variants

In [4]:
predictions = test_raw[
    [c for c in ["ID", "lot_url", "source_file", "source_quarter", "sale_date", TARGET_COL] if c in test_raw]
].copy()
predictions = predictions.rename(columns={TARGET_COL: "actual_price_usd"})

overall_rows = []
quarter_rows = []
bucket_rows = []
history_frames = []
coef_frames = []

for include_estimates, suffix, label in [
    (False, "no_estimates", "Watch Ridge - No Estimates"),
    (True, "with_estimates", "Watch Ridge - With Log Estimates"),
]:
    train_frame, feature_cols, categorical_cols, numeric_cols = make_model_frame(
        train_raw,
        target_col=TARGET_COL,
        include_estimates=include_estimates,
        verbose=VERBOSE,
    )
    test_frame, _, _, _ = make_model_frame(
        test_raw,
        target_col=TARGET_COL,
        include_estimates=include_estimates,
        verbose=VERBOSE,
    )

    schema = fit_feature_schema(
        train_frame,
        feature_cols,
        categorical_cols,
        min_category_count=MIN_CATEGORY_COUNT,
        verbose=VERBOSE,
    )
    x_train = apply_feature_schema(train_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE)
    x_test = apply_feature_schema(test_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE)

    model = fit_ridge_log_price_model(
        x_train,
        train_frame["log_target_price_usd"],
        categorical_cols=categorical_cols,
        numeric_cols=numeric_cols,
        validation_fraction=VALIDATION_FRACTION,
        random_state=RANDOM_STATE,
        verbose=VERBOSE,
    )
    pred = predict_price_from_log_model(model, x_test)
    predictions[f"ridge_pred_{suffix}_usd"] = pred

    overall_rows.append(evaluate_price_predictions(test_frame["target_price_usd"], pred, label=label))
    quarter_rows.append(evaluate_by_quarter(test_frame, pred, actual_col="target_price_usd", label=label))
    bucket_rows.append(evaluate_by_price_bucket(test_frame, pred, actual_col="target_price_usd", label=label))

    feature_report = build_feature_report(train_frame, feature_cols, categorical_cols, schema=schema)
    feature_report.to_csv(OUTPUT_DIR / f"watch_ridge_feature_report_{suffix}.csv", index=False)

    history = ridge_training_history(model)
    if not history.empty:
        history.insert(0, "model", label)
        history_frames.append(history)

    coefs = get_top_coefficients(model, top_n=40)
    coefs.insert(0, "model", label)
    coef_frames.append(coefs)

overall = pd.DataFrame(overall_rows)
quarter = pd.concat(quarter_rows, ignore_index=True)
bucket = pd.concat(bucket_rows, ignore_index=True)
overall

[watch features] Building watch model frame from 16,581 rows using target_col='ah_price_usd', include_estimates=False
[watch features] Filtered invalid/non-positive targets: 16,581->16,581
[watch features] Engineered 28 features (14 categorical, 14 numeric).
[watch features] Building watch model frame from 4,995 rows using target_col='ah_price_usd', include_estimates=False
[watch features] Filtered invalid/non-positive targets: 4,995->4,995
[watch features] Engineered 28 features (14 categorical, 14 numeric).
[watch features] Fitting watch feature schema with min_category_count=20
[watch features]   categorical auction_house: 7 kept levels from 5 observed
[watch features]   categorical location: 9 kept levels from 8 observed
[watch features]   categorical brand_clean: 52 kept levels from 457 observed
[watch features]   categorical model_clean: 86 kept levels from 6,193 observed
[watch features]   categorical ref_no_clean: 61 kept levels from 6,468 observed
[watch features]   categorica

,model,n,MAE,RMSE,MAPE,R2
0,Watch Ridge - No Estimates,4995,46160.308059,213591.325961,1.169960,0.049451
1,Watch Ridge - With Log Estimates,4995,25655.830256,176255.881422,1.073717,0.352716


## Write Outputs

In [5]:
predictions.to_csv(OUTPUT_DIR / "watch_ridge_test_predictions.csv", index=False)
overall.to_csv(OUTPUT_DIR / "watch_ridge_overall_metrics.csv", index=False)
quarter.to_csv(OUTPUT_DIR / "watch_ridge_quarter_metrics.csv", index=False)
bucket.to_csv(OUTPUT_DIR / "watch_ridge_price_bucket_metrics.csv", index=False)
if history_frames:
    pd.concat(history_frames, ignore_index=True).to_csv(OUTPUT_DIR / "watch_ridge_training_history.csv", index=False)
pd.concat(coef_frames, ignore_index=True).to_csv(OUTPUT_DIR / "watch_ridge_top_coefficients.csv", index=False)

print(f"Wrote outputs to {OUTPUT_DIR}")

Wrote outputs to /srv/notebooks/[6.0] ML Pricers/Linear_Regression/watch/Ridge_Regression/outputs
